# 01 - Preparar paths destino para datastore

Primera etapa del flujo Geosupport. Este notebook recibe una carpeta raiz con imagenes nuevas, calcula el sector por cruce espacial contra el indice de vuelos, arma el nombre oficial y define el `Path_Destino` en el datastore.

No copia archivos, no carga al mosaico y no modifica datos. El resultado principal para revisar es `04_ready_for_datastore.csv`.

In [ ]:
from datetime import datetime
from pathlib import Path
import importlib

import pandas as pd

import core.mosaic_image_audit as mosaic_audit
mosaic_audit = importlib.reload(mosaic_audit)
from core.mosaic_image_audit import *

print('Modulo auditoria:', mosaic_audit.__file__)
print('Version logica:', AUDIT_LOGIC_VERSION)

## Parametros

Cambiar solo `PATH_INPUT_IMAGENES` para una nueva entrega. El resto deja fija la logica validada del proyecto.

In [ ]:
# Carpeta raiz donde llegan las imagenes nuevas. La busqueda es recursiva.
PATH_INPUT_IMAGENES = r"\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_Drone_Sin_Procesar\INPUT\20260519_Geosupport"

# Feature class con los footprints/sectores usados para definir el nombre geografico.
PATH_FC_INDICE_VUELOS_IMGS = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\CL_MLP_PAO_v1.gdb\CL_MLP_PAO_06_COMPLEMENTOS\CL_MLP_PAO_Indice_Vuelos_PAO_IMGS_PO"
SECTOR_FIELD_INDICE_VUELOS = "Sector"
QUERY_INDICE_VUELOS = "Sensor <> 'DJI MATRICE 350 RTK'"

# Raiz del datastore donde quedaran las imagenes copiadas en la etapa siguiente.
PATH_DATASTORE_DESTINO_RAIZ = r"\\amssclgis10.ams.gmams.cl\CL_MLP_PAO"

# Salida local de auditoria/manifiesto.
run_timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_DIR = Path.cwd() / 'outputs' / 'etapa_01_preparar_paths_datastore'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Input imagenes:', PATH_INPUT_IMAGENES)
print('Feature sectores:', PATH_FC_INDICE_VUELOS_IMGS)
print('Destino datastore raiz:', PATH_DATASTORE_DESTINO_RAIZ)
print('Campo sector:', SECTOR_FIELD_INDICE_VUELOS)
print('Query sectores:', QUERY_INDICE_VUELOS)
print('Salida:', OUTPUT_DIR)

## Reglas de carpeta destino

El nombre de archivo se define por fecha + sector geografico. La carpeta destino se define por sector. Cuando un archivo puntual necesita una excepcion, se resuelve primero por nombre original.

In [ ]:
DESTINATION_FOLDER_BY_SECTOR = {
    'Camino_Alternativo_Salamanca': 'El_Mauro_Drone',
    'DME9-PA12-IIFF8': 'Chacay_El_Mauro_Drone',
    'EBD': 'El_Mauro_Drone',
    'ED1': 'Chacay_El_Mauro_Drone',
    'ED2': 'Chacay_El_Mauro_Drone',
    'EDT': 'Puerto_Punta_Chungo_Drone',
    'EM1': 'Chacay_Drone',
    'EM2_S2': 'Chacay_El_Mauro_Drone',
    'EM3': 'Chacay_El_Mauro_Drone',
    'Estacion_Cabecera': 'Chacay_Drone',
    'Estacion_Intermedia': 'Chacay_El_Mauro_Drone',
    'EV1': 'Chacay_El_Mauro_Drone',
    'EV2': 'El_Mauro_Puerto_Punta_Chungo_Drone',
    'Helipuerto': 'El_Mauro_Drone',
    'MonteAranda-NSTC-Km-84p2-a-82p3': 'Chacay_El_Mauro_Drone',
    'Patio-19B-y-Armado': 'Chacay_El_Mauro_Drone',
    'Subestacion-El-Mauro': 'El_Mauro_Drone',
    'Subestacion-El-Mauro_A_E35': 'El_Mauro_Drone',
    'TORRE_E85_A_E_125': 'Chacay_El_Mauro_Drone',
    'TORRES_E31_A_E48_PV4': 'Chacay_El_Mauro_Drone',
    'TORRES_E48_A_E84_PV4': 'Chacay_El_Mauro_Drone',
}

DESTINATION_FOLDER_BY_FILE_NAME = {
    'GEOSP-TRN-002603_ORTOFOTO_CORTADA_EM2_100526.tif': 'Chacay_El_Mauro_Drone',
    'GEOSP-TRN-002615_GS_ORTOFOTO_ESTACION DE MONITOREO NÂ°2_13-05-2026.tif': 'El_Mauro_Drone',
    'GEOSP-TRN-002617_GS_ORTOFOTO_SUBESTACION_EL MAURO_PRIORIDAD 1_13_05_26.tif': 'Chacay_El_Mauro_Drone',
    'GEOSP-TRN-002545_GS_ORTOFOTO_EB3_06-05-26.tif': 'El_Mauro_Drone',
    'GEOSP-TRN-002546_GS_ORTOFOTO_SSEE_06-05-26.tif': 'El_Mauro_Drone',
    'GEOSP-TRN-002621_GS_ORTOFOTO_ESTACION DE BOMBEO NÂº3_13_05_26.tif': 'El_Mauro_Drone',
}

display(pd.DataFrame(
    [{'Sector': key, 'Carpeta_Destino': value} for key, value in DESTINATION_FOLDER_BY_SECTOR.items()]
).sort_values('Sector').reset_index(drop=True))

## 1. Buscar imagenes

La busqueda es recursiva. Para el mosaico solo se preparan archivos `tif` y `tiff`.

In [ ]:
input_images_df = scan_input_images(PATH_INPUT_IMAGENES)
ortho_images_df = input_images_df[input_images_df['extension'].isin(ORTHO_MOSAIC_EXTENSIONS)].copy()

print(f'Archivos encontrados: {len(input_images_df)}')
print(f'Imagenes TIF/TIFF para evaluar: {len(ortho_images_df)}')

display(input_images_df.groupby('extension').size().reset_index(name='count'))
display(ortho_images_df[['file_name', 'relative_path', 'size_mb', 'modified_at']].head(20))

## 2. Calcular sector geografico y nombre esperado

El sector viene solo del cruce espacial. Si una imagen cruza mas de un sector, se usa el sector con mayor porcentaje de interseccion. El texto del sector se conserva desde el feature class y solo se reemplazan espacios por `_`.

In [ ]:
spatial_matches_df = calculate_spatial_sector_matches(
    ortho_images_df,
    PATH_FC_INDICE_VUELOS_IMGS,
    sector_field=SECTOR_FIELD_INDICE_VUELOS,
    where_clause=QUERY_INDICE_VUELOS,
)

prepared_df = add_expected_names_with_spatial_sector(
    ortho_images_df,
    spatial_matches_df,
)

display(spatial_matches_df['spatial_status'].value_counts(dropna=False).reset_index(name='count').rename(columns={'index': 'spatial_status'}))
display(prepared_df['rename_status'].value_counts(dropna=False).reset_index(name='count').rename(columns={'index': 'rename_status'}))

## 3. Preparar manifest de paths destino

`ready_for_datastore` indica imagenes con nombre y destino completos. Si hay nombres repetidos, se agrega una secuencia `-1`, `-2`, etc. al final del nombre.

In [ ]:
manifest_df = resolve_duplicate_expected_names(prepared_df)

manifest_df['destination_date_folder'] = manifest_df['expected_date_token'].map(
    lambda value: '_'.join(str(value).split('_')[:2]) if pd.notna(value) and value else None
)

destination_folder_lookup = {normalize_key(key): value for key, value in DESTINATION_FOLDER_BY_SECTOR.items()}
destination_file_lookup = {normalize_key(key): value for key, value in DESTINATION_FOLDER_BY_FILE_NAME.items()}

def resolve_destination_folder(row):
    file_folder = destination_file_lookup.get(normalize_key(row.get('file_name')))
    if file_folder:
        return file_folder
    return destination_folder_lookup.get(normalize_key(row.get('expected_sector')))

def build_destination_path(row):
    if pd.isna(row.get('destination_folder')) or pd.isna(row.get('destination_date_folder')) or pd.isna(row.get('expected_file_name')):
        return None
    return str(Path(PATH_DATASTORE_DESTINO_RAIZ) / str(row['destination_folder']) / str(row['destination_date_folder']) / str(row['expected_file_name']))

def build_review_reason(row):
    reasons = []
    if row.get('rename_status') != 'ok':
        reasons.append(str(row.get('rename_status')))
    if row.get('rename_status') == 'ok' and not row.get('destination_folder'):
        reasons.append('sin_regla_carpeta_destino')
    if row.get('duplicate_was_resolved'):
        reasons.append('nombre_duplicado_resuelto')
    if row.get('spatial_overlap_count', 0) and row.get('spatial_overlap_count', 0) > 1:
        reasons.append('cruza_multiples_sectores')
    return '|'.join(reasons) if reasons else None

manifest_df['destination_folder'] = manifest_df.apply(resolve_destination_folder, axis=1)
manifest_df['destination_path'] = manifest_df.apply(build_destination_path, axis=1)
manifest_df['ready_for_datastore'] = manifest_df['rename_status'].eq('ok') & manifest_df['destination_path'].notna()
manifest_df['review_reason'] = manifest_df.apply(build_review_reason, axis=1)

output_columns = [
    'ready_for_datastore',
    'review_reason',
    'path',
    'relative_path',
    'file_name',
    'expected_file_name',
    'destination_path',
    'original_expected_file_name',
    'expected_name',
    'expected_date_token',
    'destination_folder',
    'destination_date_folder',
    'expected_sector',
    'sector_source',
    'rename_status',
    'spatial_status',
    'spatial_sector_raw',
    'spatial_sector',
    'spatial_overlap_pct',
    'spatial_overlap_count',
    'spatial_all_matches',
    'duplicate_expected_file_name',
    'duplicate_sequence',
    'duplicate_was_resolved',
    'size_mb',
    'modified_at',
]
output_columns = [column for column in output_columns if column in manifest_df.columns]
manifest_output_df = manifest_df[output_columns].copy()

ready_df = manifest_output_df[manifest_output_df['ready_for_datastore']].copy()
review_df = manifest_output_df[~manifest_output_df['ready_for_datastore']].copy()

print(f'Listas para validar/copiar: {len(ready_df)}')
print(f'Requieren revision: {len(review_df)}')
print(f'Nombres duplicados resueltos con secuencia: {int(manifest_output_df["duplicate_was_resolved"].sum())}')

display(manifest_output_df.head(30))
display(review_df.head(30))

## 4. Exportar resultados

Usar `04_ready_for_datastore.csv` como entrada de la etapa de copia/carga, despues de revisar que `destination_path` este correcto.

In [ ]:
summary_df = pd.DataFrame([
    {'metric': 'run_timestamp', 'value': run_timestamp},
    {'metric': 'audit_logic_version', 'value': AUDIT_LOGIC_VERSION},
    {'metric': 'input_folder', 'value': PATH_INPUT_IMAGENES},
    {'metric': 'sector_feature_class', 'value': PATH_FC_INDICE_VUELOS_IMGS},
    {'metric': 'datastore_destination_root', 'value': PATH_DATASTORE_DESTINO_RAIZ},
    {'metric': 'sector_field', 'value': SECTOR_FIELD_INDICE_VUELOS},
    {'metric': 'sector_query', 'value': QUERY_INDICE_VUELOS},
    {'metric': 'input_files_count', 'value': len(input_images_df)},
    {'metric': 'ortho_images_count', 'value': len(ortho_images_df)},
    {'metric': 'ready_for_datastore_count', 'value': len(ready_df)},
    {'metric': 'review_required_count', 'value': len(review_df)},
    {'metric': 'duplicate_original_expected_file_name_count', 'value': int(manifest_output_df['duplicate_expected_file_name'].sum())},
    {'metric': 'duplicate_expected_file_name_resolved_count', 'value': int(manifest_output_df['duplicate_was_resolved'].sum())},
])

for status, count in spatial_matches_df['spatial_status'].value_counts(dropna=False).items():
    summary_df.loc[len(summary_df)] = {'metric': f'spatial_status_{status}', 'value': int(count)}

for status, count in manifest_df['rename_status'].value_counts(dropna=False).items():
    summary_df.loc[len(summary_df)] = {'metric': f'rename_status_{status}', 'value': int(count)}

summary_csv = OUTPUT_DIR / '00_summary.csv'
input_csv = OUTPUT_DIR / '01_input_images.csv'
spatial_csv = OUTPUT_DIR / '02_spatial_matches.csv'
manifest_csv = OUTPUT_DIR / '03_manifest_paths_datastore.csv'
ready_csv = OUTPUT_DIR / '04_ready_for_datastore.csv'
review_csv = OUTPUT_DIR / '05_review_required.csv'

summary_df.to_csv(summary_csv, index=False, encoding='utf-8-sig')
input_images_df.to_csv(input_csv, index=False, encoding='utf-8-sig')
spatial_matches_df.to_csv(spatial_csv, index=False, encoding='utf-8-sig')
manifest_output_df.to_csv(manifest_csv, index=False, encoding='utf-8-sig')
ready_df.to_csv(ready_csv, index=False, encoding='utf-8-sig')
review_df.to_csv(review_csv, index=False, encoding='utf-8-sig')

display(summary_df)
print('Outputs exportados en:', OUTPUT_DIR)
print('Manifest completo:', manifest_csv)
print('CSV listo para etapa siguiente:', ready_csv)
print('CSV de revision:', review_csv)